# 🏛️ ĐỒ ÁN MÔN HỌC DAT301m - STAGE 0: BASELINE MODEL & DATA AUDITING
## Huấn luyện Baseline trên dữ liệu thô ban đầu & Chẩn đoán 4 dị tật dữ liệu

* **Mục tiêu:** Thiết lập điểm mốc hiệu năng (Baseline) bằng mô hình VGG16 tự xây dựng từ đầu (16 layers scratch) trên 2.880 ảnh gốc chưa can thiệp.
* **Quy tắc:** Giữ cố định 100% kiến trúc mạng và siêu tham số ($lr=10^{-4}$ Adam, batch 32, epochs 15).


In [ ]:
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("🚀 Đang chạy trên Google Colab")
except ImportError:
    IN_COLAB = False
    print("💻 Đang chạy trên Local Machine")

if IN_COLAB:
    if os.path.exists("/content/deeplearningai-data-centric-competition"):
        os.chdir("/content/deeplearningai-data-centric-competition")
    elif not os.path.exists("data"):
        os.system("git clone https://github.com/Nov05/deeplearningai-data-centric-competition.git")
        if os.path.exists("deeplearningai-data-centric-competition"):
            os.chdir("deeplearningai-data-centric-competition")

# Giải nén data và label_book nếu chưa có
import tarfile
if not os.path.exists("data") and os.path.exists("original/data.tar.gz"):
    with tarfile.open("original/data.tar.gz", "r:gz") as tar:
        tar.extractall(".")
    print("✅ Đã giải nén dữ liệu data thành công.")

if not os.path.exists("label_book") and os.path.exists("original/label_book.tar.gz"):
    with tarfile.open("original/label_book.tar.gz", "r:gz") as tar:
        tar.extractall(".")
    print("✅ Đã giải nén dữ liệu label_book thành công.")

import glob
raw_count = len(glob.glob("data/**/*.png", recursive=True))
print(f"✅ Dữ liệu gốc: Tìm thấy {raw_count} ảnh.")
assert raw_count > 0, "LỖI: Chưa có ảnh trong thư mục 'data/'!"


🚀 Đang chạy trên Google Colab


/tmp/ipykernel_4213/3864034174.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(".")


✅ Đã giải nén dữ liệu data thành công.
✅ Dữ liệu gốc: Tìm thấy 2880 ảnh.


## 1. KHỞI TẠO KIẾN TRÚC VGG-16 TỰ XÂY DỰNG (SCRATCH)
> ⚠️ **Quy tắc của Giảng viên:** Không dùng thư viện `keras.applications.VGG16`, không dùng pretrained weights. Toàn bộ 16 layers (13 Conv + 3 Dense) được code tay từ đầu.


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import hashlib
import shutil
from PIL import Image

IMAGE_SIZE = (32, 32)
BATCH_SIZE = 32
NUM_CLASSES = 10
CLASS_NAMES = ["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"]
RANDOM_SEED = 123
EPOCHS = 15

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"🚀 Thiết bị tăng tốc: ĐÃ BẬT GPU ({gpus[0].name}) -> Tốc độ huấn luyện cực nhanh!")
else:
    print("⚠️ Thiết bị hiện tại: CPU (Khuyên dùng: Trên Colab vào Menu Runtime -> Change runtime type -> Chọn T4 GPU)")

def build_fixed_vgg16_model():
    """
    KIẾN TRÚC VGG16 TỰ XÂY DỰNG TỪ ĐẦU (FROM SCRATCH) THEO YÊU CẦU GIẢNG VIÊN:
    - KHÔNG dùng thư viện có sẵn tf.keras.applications.VGG16
    - KHÔNG dùng pretrained weights ImageNet
    - Tự code tay đúng 13 lớp Conv2D + 5 lớp MaxPooling2D + 3 lớp Dense = 16 Layers VGG16
    - Khởi tạo He-Normal, cố định bất biến qua mọi Stage
    """
    tf.random.set_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    inputs = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name="input_image")
    x = tf.keras.layers.Rescaling(1.0 / 255.0, name="rescaling")(inputs)

    # Block 1 (2 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b1_c1")(x)
    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b1_c2")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b1_pool")(x)

    # Block 2 (2 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b2_c1")(x)
    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b2_c2")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b2_pool")(x)

    # Block 3 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c1")(x)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c2")(x)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b3_pool")(x)

    # Block 4 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c1")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c2")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b4_pool")(x)

    # Block 5 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c1")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c2")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b5_pool")(x)

    # Classifier Head (13 Conv + 3 Dense = 16 Layers VGG16)
    x = tf.keras.layers.Flatten(name="flatten")(x)
    x = tf.keras.layers.Dense(512, activation="relu", kernel_initializer="he_normal", name="fc1")(x)
    x = tf.keras.layers.Dropout(0.5, name="dropout1")(x)
    x = tf.keras.layers.Dense(512, activation="relu", kernel_initializer="he_normal", name="fc2")(x)
    x = tf.keras.layers.Dropout(0.5, name="dropout2")(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Fixed_VGG16_Scratch")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

print("✅ Đã khởi tạo cấu trúc Fixed VGG16 Scratch thành công (15.245.130 tham số).")


TensorFlow Version: 2.21.0
🚀 Thiết bị tăng tốc: ĐÃ BẬT GPU (/physical_device:GPU:0) -> Tốc độ huấn luyện cực nhanh!
✅ Đã khởi tạo cấu trúc Fixed VGG16 Scratch thành công (15.245.130 tham số).


## 2. NẠP DỮ LIỆU THÔ BAN ĐẦU (RAW DATA: TRAIN + VAL + LABEL BOOK)


In [ ]:
train_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    "data/train", labels="inferred", label_mode="categorical",
    class_names=CLASS_NAMES, shuffle=True, seed=RANDOM_SEED,
    batch_size=BATCH_SIZE, image_size=IMAGE_SIZE
)

val_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    "data/val", labels="inferred", label_mode="categorical",
    class_names=CLASS_NAMES, shuffle=False, seed=RANDOM_SEED,
    batch_size=BATCH_SIZE, image_size=IMAGE_SIZE
)

lb_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    "label_book", labels="inferred", label_mode="categorical",
    class_names=CLASS_NAMES, shuffle=False,
    batch_size=BATCH_SIZE, image_size=IMAGE_SIZE
)

print(f"Số batch Train: {train_ds_raw.cardinality().numpy()} | Val: {val_ds_raw.cardinality().numpy()}")


Found 2067 files belonging to 10 classes.
Found 813 files belonging to 10 classes.
Found 52 files belonging to 10 classes.
Số batch Train: 65 | Val: 26


## 3. HUẤN LUYỆN BASELINE TRONG 15 EPOCHS


In [ ]:
baseline_model = build_fixed_vgg16_model()

history_s0 = baseline_model.fit(
    train_ds_raw,
    validation_data=val_ds_raw,
    epochs=EPOCHS,
    verbose=1
)

val_loss_s0, val_acc_s0 = baseline_model.evaluate(val_ds_raw, verbose=0)
lb_loss_s0, lb_acc_s0 = baseline_model.evaluate(lb_ds_raw, verbose=0)

print("="*60)
print(f"📊 KẾT QUẢ STAGE 0 (BASELINE):")
print(f"  • Validation Accuracy (Nội bộ): {val_acc_s0*100:.2f}% | Loss: {val_loss_s0:.4f}")
print(f"  • Label Book Accuracy (Test độc lập): {lb_acc_s0*100:.2f}% | Loss: {lb_loss_s0:.4f}")
print("="*60)


Epoch 1/15
65/65 ━━━━━━━━━━━━━━━━━━━━ 63s 608ms/step - accuracy: 0.1185 - loss: 2.3437 - val_accuracy: 0.1033 - val_loss: 2.3040
Epoch 2/15
65/65 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.1190 - loss: 2.2930 - val_accuracy: 0.1033 - val_loss: 2.3034
Epoch 3/15
65/65 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.1272 - loss: 2.2923 - val_accuracy: 0.1033 - val_loss: 2.3046
Epoch 4/15
65/65 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.1330 - loss: 2.2918 - val_accuracy: 0.1193 - val_loss: 2.2976
Epoch 5/15
65/65 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step - accuracy: 0.1805 - loss: 2.2477 - val_accuracy: 0.1070 - val_loss: 2.2601
Epoch 6/15
65/65 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.2051 - loss: 2.1412 - val_accuracy: 0.2448 - val_loss: 1.9304
Epoch 7/15
65/65 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.2850 - loss: 1.9058 - val_accuracy: 0.3223 - val_loss: 1.7228
Epoch 8/15
65/65 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.3614 - loss: 1.6841 - val_accuracy: 0.3358 -

## 4. CHẨN ĐOÁN 4 KHIẾM KHUYẾT CỐT TỬ CỦA TẬP DỮ LIỆU GỐC


In [ ]:
# 1. Quét rò rỉ dữ liệu qua MD5
train_hashes = {}
for f in glob.glob("data/train/*/*.png"):
    with open(f, "rb") as fp:
        train_hashes[hashlib.md5(fp.read()).hexdigest()] = f

leakage_count = 0
for f in glob.glob("data/val/*/*.png"):
    with open(f, "rb") as fp:
        h = hashlib.md5(fp.read()).hexdigest()
        if h in train_hashes:
            leakage_count += 1
print(f"⚠️ Phát hiện {leakage_count} ảnh trùng lặp 100% giữa Train và Validation (Data Leakage)!")

# 2. Quét ảnh trắng tinh
blank_count = 0
for f in glob.glob("data/*/*/*.png"):
    arr = np.array(Image.open(f).convert("L"))
    if arr.min() == 255:
        print(f"⚠️ Phát hiện ảnh trắng xóa: {f}")
        blank_count += 1


⚠️ Phát hiện 49 ảnh trùng lặp 100% giữa Train và Validation (Data Leakage)!
⚠️ Phát hiện ảnh trắng xóa: data/train/vi/aac26b40-ce5d-11eb-b317-38f9d35ea60f.png
